# IDSSE 03 — video review sheet: 1. FC Köln vs FC Bayern (27 May 2023)

Blind check of the variant B possession labels (notebook 02) against video of another game, which
RULES.md allows ("hand-labelled data from other games is fine"; document what, how, by whom, how much).

Three kinds of moment are sampled, then shuffled so the reviewer cannot tell them apart:

| kind | what it is | what a good label rule needs |
|---|---|---|
| `dropped` | the tracking flag changed team, variant B ignored it (no control action by the new team) | mostly **no** |
| `missed` | the event log has a turnover, variant B has no team change there | mostly **no** (else B misses real ones) |
| `kept` | the flag changed team and the event log confirms it | mostly **yes** |

Every row asks the same question: *did this team gain controlled possession here?* The kind of each row
goes to a separate key file in `cache/` (gitignored) and is only joined after the review.

Outputs: `eda/review/J03WMX_review_sheet.csv` (fill in `answer`, `notes`) and
`cache/idsse/J03WMX_review_key.csv`.

In [1]:
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "eda" else os.getcwd()
sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
from src.idsse import parse, labels, events

FPS, HOME_VALUE, MID = parse.FPS, 1, "J03WMX"
N_PER_KIND = {"dropped": 12, "missed": 8, "kept": 10}
MIN_GAP_S = 15                      # moments at least this far apart, so each is a separate clip
RNG = np.random.default_rng(20261115)

f = parse.files()[MID]
info = parse.read_matchinfo(f["info"])
m = parse.load_match(MID, quiet=True)
ev = events.read_events(f["events"], info)
off, _, _ = events.estimate_offset(m, ev)
ctl = events.control_events(ev, m, off)
raw = labels.frame_codes(m, HOME_VALUE)
tov = events.event_turnovers(ctl, raw)
varB = events.event_confirmed_relabel(raw, m["half"], tov)
TEAM = {0: info["home_name"], 1: info["away_name"]}
print(TEAM, "| event turnovers", len(tov))

{0: '1. FC Köln', 1: 'FC Bayern München'} | event turnovers 133


## Match clock

Kickoff of each half = first live frame (`BallStatus` alive). Clock = time since that kickoff, plus 45:00
in the second half, as on the broadcast clock. The goals are listed as anchors: check one against the
video first and shift all times by the same amount if the broadcast clock differs.

In [2]:
half = m["half"]
kick = {h: np.flatnonzero((half == h) & (m["status"] == 1))[0] for h in np.unique(half)}

def clock(row):
    h = half[row]
    s = (row - kick[h]) / FPS + (45 * 60 if h == 2 else 0)
    return f"{int(s // 60):02d}:{int(s % 60):02d}"

import xml.etree.ElementTree as ET
goals = []
for e in ET.parse(f["events"]).getroot().iter("Event"):
    shot = e.find(".//ShotAtGoal")
    if shot is not None and shot.find("SuccessfulShot") is not None:
        r = int(events.event_rows(m, [pd.Timestamp(e.get("EventTime")).tz_convert(None)], off)[0])
        goals.append({"half": int(half[r]), "clock": clock(r),
                      "team": TEAM[info["teams"][shot.get("Team")]["idx"]]})
pd.DataFrame(goals)

,half,clock,team
0,1,07:48,FC Bayern München
1,2,80:03,1. FC Köln
2,2,88:14,FC Bayern München


## Sample the moments

In [3]:
lt = events.label_turnovers(raw, half)
confirmed = events._confirmed(lt, tov, int(0.5 * FPS))
ltB = events.label_turnovers(varB, half)

cand = []
for (row, team), ok in zip(lt, confirmed):
    cand.append({"row": int(row), "team": int(team), "kind": "kept" if ok else "dropped"})
for a, b, team in tov:
    lo, hi = np.searchsorted(ltB[:, 0], [a - int(0.5 * FPS), b + int(0.5 * FPS) + 1])
    if not (ltB[lo:hi, 1] == team).any():
        cand.append({"row": int(b), "team": int(team), "kind": "missed"})
cand = pd.DataFrame(cand)
print(cand.kind.value_counts().to_dict())

picked = []
for kind, n in N_PER_KIND.items():
    pool = cand[cand.kind == kind].sample(frac=1, random_state=int(RNG.integers(1e9)))
    taken = 0
    for r in pool.itertuples(index=False):
        if taken == n:
            break
        if all(abs(r.row - p["row"]) >= MIN_GAP_S * FPS for p in picked):
            picked.append(r._asdict())
            taken += 1
picked = pd.DataFrame(picked).sort_values("row").reset_index(drop=True)
picked["id"] = np.arange(1, len(picked) + 1)   # ordered by time, so the kind cannot be guessed from the id
picked.kind.value_counts()

{'dropped': 278, 'kept': 130, 'missed': 18}


kind
dropped    12
kept       10
missed      8
Name: count, dtype: int64

## Where on the pitch

Described from the defending team's point of view, using where each team's keeper stands in that half.

In [4]:
gk_x = {}
for h in np.unique(half):
    for t in (0, 1):
        j = np.flatnonzero(m["person_gk"] & (m["person_team"] == t))
        gk_x[(h, t)] = np.nanmean(m["xy"][half == h][:, j, 0])

def where(row):
    h = half[row]
    x = m["ball"][row, 0]
    if not np.isfinite(x):
        return ""
    if abs(x) < 17.5:
        return "around halfway"
    own = TEAM[0] if np.sign(x) == np.sign(gk_x[(h, 0)]) else TEAM[1]   # whose goal this end is
    return f"{own}'s half, " + ("near their box" if abs(x) > 36 else "middle of their half")

sheet = pd.DataFrame({
    "id": picked.id,
    "half": [int(half[r]) for r in picked.row],
    "clock": [clock(r) for r in picked.row],
    "question": [f"Did {TEAM[t]} gain controlled possession here?" for t in picked.team],
    "where": [where(r) for r in picked.row],
    "answer": "", "notes": "",
})
os.makedirs(os.path.join(ROOT, "eda", "review"), exist_ok=True)
sheet.to_csv(os.path.join(ROOT, "eda", "review", f"{MID}_review_sheet.csv"), index=False)
picked[["id", "kind", "row", "team"]].to_csv(os.path.join(parse.CACHE_DIR, f"{MID}_review_key.csv"), index=False)
sheet[["id", "half", "clock", "question", "where"]]

,id,half,clock,question,where
0,1,1,03:46,Did 1. FC Köln gain controlled possession here?,"1. FC Köln's half, middle of their half"
1,2,1,04:34,Did FC Bayern München gain controlled possessi...,"1. FC Köln's half, middle of their half"
2,3,1,05:36,Did 1. FC Köln gain controlled possession here?,around halfway
3,4,1,06:11,Did FC Bayern München gain controlled possessi...,"1. FC Köln's half, middle of their half"
4,5,1,06:56,Did 1. FC Köln gain controlled possession here?,"1. FC Köln's half, middle of their half"
5,6,1,09:26,Did 1. FC Köln gain controlled possession here?,"1. FC Köln's half, near their box"
6,7,1,10:32,Did 1. FC Köln gain controlled possession here?,around halfway
7,8,1,11:58,Did FC Bayern München gain controlled possessi...,"FC Bayern München's half, middle of their half"
8,9,1,13:52,Did 1. FC Köln gain controlled possession here?,"FC Bayern München's half, middle of their half"
9,10,1,14:15,Did FC Bayern München gain controlled possessi...,around halfway


## How to answer

Watch each clock time from about 5 s before to 5 s after. In `answer` write one of:

- `Y` the team named gained control (kept the ball, passed it on, shot, dribbled);
- `T` only a touch / deflection / block, the other team kept or won it straight back;
- `P` the ball went past (no touch), or the moment is a normal pass between team-mates of the other team;
- `?` can't see it (replay, camera cut away, not sure).

The scoring (join with the key, agreement per kind) is added to this notebook once the sheet is filled in.